# Facial Recognition Technology (FRT) Ethics Prototype

This notebook implements a simplified decision pipeline designed to validate
both functional and ethical goals of a facial recognition system.

The implementation focuses on three ethical cases:

1. **Deontological constraint (fairness and rights)**
   - Prevent configurations that introduce disproportionate demographic harm

2. **Utilitarian adjustment (severity-based configuration)**
   - Adjust threshold and return limits based on crime severity

3. **Human-in-the-loop enforcement (virtue ethics)**
   - Ensure system outputs remain advisory and require human review

The goal is not to build a full system, but to demonstrate:
- how ethical constraints are encoded
- how system behaviour changes under those constraints
- when actions are allowed or explicitly withheld

In [23]:
from z3 import *
from typing import List, Dict, Any
from datetime import datetime, UTC

## Domain Elements

The system models the following variables:

- `T`: face-match threshold
- `R`: return limit
- `N`: watchlist size
- `Csev`: crime severity

These directly correspond to the configurable parameters described in the system.

In [24]:
watchlist = [
    {"person_id": "W001", "name": "Aoife Byrne"},
    {"person_id": "W002", "name": "Mohammed Rahman"},
    {"person_id": "W003", "name": "Kwame Mensah"},
    {"person_id": "W004", "name": "Siobhan Murphy"},
    {"person_id": "W005", "name": "Arjun Patel"},
]

observed_candidates = [
    {"candidate_id": "W001", "display_name": "Aoife Byrne", "similarity_score": 0.93},
    {"candidate_id": "W002", "display_name": "Mohammed Rahman", "similarity_score": 0.84},
    {"candidate_id": "W003", "display_name": "Kwame Mensah", "similarity_score": 0.79},
    {"candidate_id": "W004", "display_name": "Siobhan Murphy", "similarity_score": 0.67},
]

fpir_table = {
    0.90: {"White": 0.002, "Black": 0.003, "Asian": 0.003},
    0.85: {"White": 0.004, "Black": 0.007, "Asian": 0.006},
    0.80: {"White": 0.009, "Black": 0.017, "Asian": 0.015},
    0.75: {"White": 0.018, "Black": 0.036, "Asian": 0.031},
    0.60: {"White": 0.079, "Black": 0.152, "Asian": 0.139},
}

## Functional Validation: Candidate Search

This validates the basic system functionality:
- filtering candidates by threshold
- ordering by similarity
- limiting results

In [25]:
def normalise_threshold(T):
    return min(fpir_table.keys(), key=lambda x: abs(x - T))


def run_search(candidates, T, R):
    T = normalise_threshold(T)
    results = [c for c in candidates if c["similarity_score"] >= T]
    return sorted(results, key=lambda x: x["similarity_score"], reverse=True)[:R]

## Ethical Case 1: Deontological Constraint (Fairness)

This encodes a strict rule:
- configurations that fall below an acceptable threshold or produce high disparity
  must be blocked

This represents a **hard constraint** where action is withheld.

In this case, a satisfiable (`sat`) constraint set means the proposed configuration is ethically permissible, while an unsatisfiable (`unsat`) result means the configuration must be blocked.

In [26]:
def get_disparity(T):
    profile = fpir_table[normalise_threshold(T)]
    return max(profile.values()) - min(profile.values())


# Uses Z3 to decide whether a proposed configuration is ethically permissible.
def check_fairness(T, N):
    D = get_disparity(T)

    threshold_fail = T < 0.60
    disparity_fail = T <= 0.75 and D >= 0.015
    scaling_fail = N > 20 and T <= 0.80

    s = Solver()
    t = Real("t")
    n = Int("n")
    d = Real("d")

    s.add(t == RealVal(str(T)))
    s.add(n == N)
    s.add(d == RealVal(str(D)))

    s.add(t >= RealVal("0.60"))
    s.add(Not(And(t <= RealVal("0.75"), d >= RealVal("0.015"))))
    s.add(Not(And(n > 20, t <= RealVal("0.80"))))

    result = s.check()

    reasons = []
    if threshold_fail:
        reasons.append("Threshold below acceptable minimum.")
    elif T <= 0.60:
        reasons.append("Threshold is at the lowest permissible boundary and remains high-risk.")
    if disparity_fail:
        reasons.append("Configuration presents elevated risk of false identification.")
    if scaling_fail:
        reasons.append("Watchlist size is disproportionate for this threshold.")
    if not reasons:
        reasons.append("Fairness constraints satisfied.")

    return {
        "passed": result == sat,
        "disparity": round(D, 3),
        "blocked": result != sat,
        "reasons": reasons
    }

## Ethical Case 2: Utilitarian Adjustment (Severity)

The system adapts configuration based on severity:

- low severity → strict threshold, minimal results
- high severity → relaxed threshold, larger result set

This represents **action adjustment**, not blocking.

In [27]:
# Maps crime severity to a proposed threshold and return limit using Z3.
def adjust_for_severity(Csev):
    s = Solver()

    sev = Int("sev")
    T = Real("T")
    R = Int("R")

    s.add(sev == Csev, sev >= 1, sev <= 5)

    s.add(
        If(sev == 1, And(T == RealVal("0.90"), R == 1),
        If(sev == 2, And(T == RealVal("0.85"), R == 2),
        If(sev == 3, And(T == RealVal("0.80"), R == 3),
        If(sev == 4, And(T == RealVal("0.75"), R == 5),
                        And(T == RealVal("0.60"), R == 10)))))
    )

    result = s.check()
    if result != sat:
        raise ValueError("Csev must be between 1 and 5.")

    m = s.model()
    return float(m[T].as_decimal(10).replace("?", "")), m[R].as_long()

## Ethical Case 3: Human Review (Virtue Ethics)

The system does not produce final decisions.

Instead:
- it returns candidates
- a human operator must approve or reject

In [28]:
def review(candidate, decision):
    return {
        "candidate_id": candidate["candidate_id"],
        "candidate_name": candidate["display_name"],
        "decision": decision,
        "time": datetime.now(UTC).isoformat()
    }

## Full System Execution

This combines:
1. severity-based configuration
2. fairness validation
3. candidate search
4. human review

It also explicitly demonstrates when actions are withheld.

In [29]:
# Runs the full decision pipeline: severity adjustment, fairness validation, search, and human review.
def system_run(candidates, watchlist, Csev, decision="reject"):
    T, R = adjust_for_severity(Csev)
    fairness = check_fairness(T, len(watchlist))

    if not fairness["passed"]:
        return {
            "status": "BLOCKED",
            "reason": "Fairness constraint triggered.",
            "threshold": T,
            "return_limit": R,
            "fairness": fairness
        }

    results = run_search(candidates, T, R)

    if not results:
        return {
            "status": "NO_MATCH",
            "threshold": T,
            "return_limit": R
        }

    review_result = review(results[0], decision)

    return {
        "status": "ACTION_ALLOWED",
        "threshold": T,
        "return_limit": R,
        "results": results,
        "review": review_result
    }

## Scenario Validation

The following scenarios demonstrate:
- when actions proceed
- when they are restricted
- when they are blocked

In [30]:
print("\n=== Scenario Results ===\n")

for sev in range(1, 6):
    result = system_run(observed_candidates, watchlist, sev)

    if result["status"] == "ACTION_ALLOWED":
        print(
            f"Scenario {sev}: status={result['status']}, "
            f"T={result['threshold']}, R={result['return_limit']}, "
            f"top_candidate={result['results'][0]['display_name']}"
        )
    elif result["status"] == "BLOCKED":
        print(
            f"Scenario {sev}: status={result['status']}, "
            f"T={result['threshold']}, R={result['return_limit']}, "
            f"reasons={result['fairness']['reasons']}"
        )
    else:
        print(
            f"Scenario {sev}: status={result['status']}, "
            f"T={result['threshold']}, R={result['return_limit']}"
        )


=== Scenario Results ===

Scenario 1: status=ACTION_ALLOWED, T=0.9, R=1, top_candidate=Aoife Byrne
Scenario 2: status=ACTION_ALLOWED, T=0.85, R=2, top_candidate=Aoife Byrne
Scenario 3: status=ACTION_ALLOWED, T=0.8, R=3, top_candidate=Aoife Byrne
Scenario 4: status=BLOCKED, T=0.75, R=5, reasons=['Configuration presents elevated risk of false identification.']
Scenario 5: status=BLOCKED, T=0.6, R=10, reasons=['Threshold is at the lowest permissible boundary and remains high-risk.', 'Configuration presents elevated risk of false identification.']
